# Test der HypothekenTragbarkeitAPI
Vorher API starten: `uvicorn Code.main:app --reload` (im Projektordner).

In [1]:
import requests

# URL der lokal laufenden API
URL = "http://127.0.0.1:8000/tragbarkeit-check"

In [2]:
# Die 3 Testfälle: (Einkommen, Kaufpreis, Eigenkapital)
faelle = [
    {"einkommen_brutto_jahr": 180000, "kaufpreis": 900000, "eigenkapital": 200000},  # ca. 28.1%, tragbar
    {"einkommen_brutto_jahr": 120000, "kaufpreis": 900000, "eigenkapital": 200000},  # ca. 42.2%, nicht tragbar
    {"einkommen_brutto_jahr": 180000, "kaufpreis": 900000, "eigenkapital": 150000},  # ca. 31.4%, EK unter 20%
]

for i, fall in enumerate(faelle, 1):
    antwort = requests.post(URL, json=fall)
    print(f"Fall {i}: Status {antwort.status_code}")
    print(antwort.json(), "\n")

Fall 1: Status 200
{'hypothek': 700000.0, 'belehnung_prozent': 77.8, 'zinskosten': 35000.0, 'nebenkosten': 9000.0, 'amortisation': 6666.67, 'wohnkosten_jahr': 50666.67, 'tragbarkeit_prozent': 28.1, 'tragbar': True, 'hinweise': []} 

Fall 2: Status 200
{'hypothek': 700000.0, 'belehnung_prozent': 77.8, 'zinskosten': 35000.0, 'nebenkosten': 9000.0, 'amortisation': 6666.67, 'wohnkosten_jahr': 50666.67, 'tragbarkeit_prozent': 42.2, 'tragbar': False, 'hinweise': ['Tragbarkeit über 33%']} 

Fall 3: Status 200
{'hypothek': 750000.0, 'belehnung_prozent': 83.3, 'zinskosten': 37500.0, 'nebenkosten': 9000.0, 'amortisation': 10000.0, 'wohnkosten_jahr': 56500.0, 'tragbarkeit_prozent': 31.4, 'tragbar': False, 'hinweise': ['Eigenkapital unter 20%']} 



In [3]:
# Validierung: Eigenkapital >= Kaufpreis -> HTTP 422
antwort = requests.post(URL, json={"einkommen_brutto_jahr": 180000, "kaufpreis": 900000, "eigenkapital": 900000})
print(antwort.status_code, antwort.json())

422 {'detail': 'Eigenkapital muss kleiner als der Kaufpreis sein.'}
